# Topic 6 — Vector Databases: Practice Notebook

Topic 5 treated retrieval as `dense_search`: embed the query, take the dot
product against every stored embedding, sort, return the top-k. That
function is correct, but it hides an assumption — that "every stored
embedding" is small enough to scan in full for every query. This notebook
opens that box. Section 1 makes the brute-force cost explicit and introduces
the *recall/speed tradeoff* that every approximate nearest-neighbor (ANN)
method makes. Section 2 builds, by hand, the graph structure behind the most
common ANN algorithm, **HNSW** (Hierarchical Navigable Small World), and
traces a search through it one step at a time. Section 3 swaps the by-hand
graph for FAISS's real implementation and measures the tradeoff on a larger
synthetic dataset. Sections 4 and 5 are exercises that move the same vectors
into two real vector databases — Chroma (embedded) and Qdrant (server-based)
— and explore how metadata filtering interacts with ANN search. Section 6
closes with vector *quantization*, which shrinks the index itself,
foreshadowing Topic 9's model quantization.

In [1]:
import time

import numpy as np
import faiss

## 1. From Brute-Force to ANN

Imagine the corpus from Topic 5 grown from a handful of documents to 50,000
— still tiny by production standards, but enough to feel the cost.
**Brute-force (exact) nearest-neighbor search** computes the distance from
the query to *every* stored vector, then sorts. If there are $N$ stored
vectors of dimension $D$, that's $N \times D$ multiply-subtract operations
per query — $O(N \cdot D)$. This is exactly what `dense_search` did in
Topic 5; it was fine there because the corpus had a dozen entries.

**Approximate nearest-neighbor (ANN) search** accepts that the top-k it
returns might occasionally miss the *true* top-k (a metric called
**recall**), in exchange for not touching every vector. The dataset below
mimics a small embedding collection: five topic clusters (`physics`,
`biology`, `history`, `cooking`, `sports`), each a random center in
128-dimensional space, with individual vectors scattered around their
cluster's center by Gaussian noise — a rough but realistic stand-in for how
a sentence-embedding model places same-topic documents near each other.

In [2]:
np.random.seed(42)

N_VECTORS = 50_000
DIM = 128
CATEGORIES = ["physics", "biology", "history", "cooking", "sports"]

# Each category gets a random "center" in 128-dim space; vectors are the
# center plus Gaussian noise -- a rough stand-in for how a real embedding
# model places same-topic documents near each other.
centers = np.random.randn(len(CATEGORIES), DIM).astype("float32")
category_ids = np.random.randint(0, len(CATEGORIES), size=N_VECTORS)
noise = np.random.randn(N_VECTORS, DIM).astype("float32") * 0.5
vectors = centers[category_ids] + noise
print(f"vectors.shape = {vectors.shape}")  # (50000, 128)
print(f"Category counts: {np.bincount(category_ids)}")

# Query vectors are built the same way -- a real query embedding lands near
# one topic cluster, not at a random point in 128-dim space.
N_QUERIES = 20
K = 10
query_category_ids = np.random.randint(0, len(CATEGORIES), size=N_QUERIES)
query_noise = np.random.randn(N_QUERIES, DIM).astype("float32") * 0.5
queries = centers[query_category_ids] + query_noise
print(f"queries.shape = {queries.shape}")  # (20, 128)


def brute_force_topk(vectors, query, k=10):
    """Exact k-nearest-neighbors by L2 distance, computed with plain numpy.

    Args:
        vectors: (N, D) array of stored vectors.
        query: (D,) query vector.
        k: number of neighbors to return.
    Returns:
        Indices of the k closest vectors, sorted nearest-first.
    MATH: dist(v, q) = ||v - q||_2 for every stored vector v.
    """
    diffs = vectors - query  # (N, D)
    dists = np.linalg.norm(diffs, axis=1)  # (N,)
    return np.argsort(dists)[:k]


start = time.perf_counter()
brute_force_results = [brute_force_topk(vectors, q, k=K) for q in queries]
elapsed = time.perf_counter() - start
print(f"Brute-force search for {N_QUERIES} queries took {elapsed*1000:.2f} ms "
      f"({elapsed/N_QUERIES*1000:.4f} ms/query)")
print(f"Top-{K} neighbors for query 0: {brute_force_results[0]}")

vectors.shape = (50000, 128)
Category counts: [ 9907 10086  9836 10066 10105]
queries.shape = (20, 128)


Brute-force search for 20 queries took 259.90 ms (12.9950 ms/query)
Top-10 neighbors for query 0: [24762  4281 29230 34062 48673 34101 35600  5390 24329 45167]


At roughly 21 ms per query, brute force here means
$50{,}000 \times 128 \approx 6.4$ million multiply-subtracts just to compute
the distances, plus a sort over 50,000 values, *for every single query*. A
production system answering even a modest 50 queries per second would spend
over a full second of CPU time per second just on distance computation — and
that's at $N=50{,}000$. Real corpora run into the millions or billions of
vectors, where $O(N \cdot D)$ scanning is no longer viable at any reasonable
latency.

ANN methods avoid the full scan by pre-organizing the vectors into a
structure that lets a query "skip" most of them — at the cost of
occasionally skipping past the true nearest neighbor too. The most widely
used such structure today is **HNSW**, which Section 2 builds by hand on a
tiny example before Section 3 unleashes FAISS's real implementation on this
50,000-vector dataset.

## 2. HNSW Intuition

**HNSW (Hierarchical Navigable Small World)** organizes vectors into several
stacked graphs, called *layers*. Every vector lives in layer 0 (the bottom,
densest layer), where each vector is connected to a handful of its nearest
neighbors. A small random subset of vectors *also* appears in layer 1,
connected more sparsely; an even smaller subset appears in layer 2, and so
on. The top layer typically has just one or a few nodes.

A search starts at a fixed **entry point** in the topmost layer and performs
a **greedy walk**: look at the current node's neighbors *in this layer*,
move to whichever one is closest to the query, and repeat until no neighbor
is closer than the current node. At that point, drop down one layer —
keeping the current node as the new starting point — and repeat the greedy
walk using that layer's (denser) connections. The process ends after a
greedy walk on layer 0, the densest layer, which yields the final
approximate answer.

The intuition is a road network. Layer 2 is the interstate highway system —
a handful of long-range connections that let you cross the country in a few
hops. Layer 1 is state highways — denser, shorter-range. Layer 0 is local
streets — every house connected to its immediate neighbors. To get from one
specific house to another, you don't drive local streets the whole way: you
take local streets to a highway on-ramp, the highway most of the distance,
then local streets again at the destination. HNSW search does the same thing
in reverse — starting on the highway (layer 2, where the entry point lives)
and exiting onto progressively more local roads as it gets close to the
query.

Below is a tiny 6-point graph in 2D, built by hand using exactly this
layered structure, with a query point $Q = (7.5, 1.5)$.

```
|                                        |
|                                    C   |
|                                        |
|                                        |
|                                        |
|                                        |
|                                        |
|                                        |
|                                        |
|    B               D                   |
|                                        |
|                                        |
|                                        |
|                                        |
|                                        |
|                                E       |
|                              Q         |
|                                        |
|                        F               |
|A                                       |
+----------------------------------------+
  0                                  x=9
```

In [3]:
POINTS = {
    "A": np.array([0.0, 0.0]),
    "B": np.array([1.0, 5.0]),
    "C": np.array([9.0, 9.0]),
    "D": np.array([5.0, 5.0]),
    "E": np.array([8.0, 2.0]),
    "F": np.array([6.0, 0.5]),
}
QUERY = np.array([7.5, 1.5])


def dist_to_query(node):
    return float(np.linalg.norm(POINTS[node] - QUERY))


print("Distance from each point to the query Q = (7.5, 1.5):")
for node in POINTS:
    print(f"  {node} = {POINTS[node]}  dist(Q) = {dist_to_query(node):.3f}")

print("\nTrue nearest neighbor (brute force), sorted by distance:")
for node in sorted(POINTS, key=dist_to_query):
    print(f"  {node}: {dist_to_query(node):.3f}")

print("\nPairwise distances between points:")
names = list(POINTS.keys())
for a in names:
    row = "  ".join(f"{a}-{b}={np.linalg.norm(POINTS[a] - POINTS[b]):.2f}" for b in names if a != b)
    print(f"  {a}: {row}")

Distance from each point to the query Q = (7.5, 1.5):
  A = [0. 0.]  dist(Q) = 7.649
  B = [1. 5.]  dist(Q) = 7.382
  C = [9. 9.]  dist(Q) = 7.649
  D = [5. 5.]  dist(Q) = 4.301
  E = [8. 2.]  dist(Q) = 0.707
  F = [6.  0.5]  dist(Q) = 1.803

True nearest neighbor (brute force), sorted by distance:
  E: 0.707
  F: 1.803
  D: 4.301
  B: 7.382
  A: 7.649
  C: 7.649

Pairwise distances between points:
  A: A-B=5.10  A-C=12.73  A-D=7.07  A-E=8.25  A-F=6.02
  B: B-A=5.10  B-C=8.94  B-D=4.00  B-E=7.62  B-F=6.73
  C: C-A=12.73  C-B=8.94  C-D=5.66  C-E=7.07  C-F=9.01
  D: D-A=7.07  D-B=4.00  D-C=5.66  D-E=4.24  D-F=4.61
  E: E-A=8.25  E-B=7.62  E-C=7.07  E-D=4.24  E-F=2.50
  F: F-A=6.02  F-B=6.73  F-C=9.01  F-D=4.61  F-E=2.50


### Building the layers from these distances

**Layer 0** (the bottom, densest layer) connects every point to its $M=2$
nearest neighbors, read directly off the pairwise distance table above:

- $A$: nearest are $B$ (5.10) and $F$ (6.02)
- $B$: nearest are $D$ (4.00) and $A$ (5.10)
- $C$: nearest are $D$ (5.66) and $E$ (7.07)
- $D$: nearest are $B$ (4.00) and $E$ (4.24)
- $E$: nearest are $F$ (2.50) and $D$ (4.24)
- $F$: nearest are $E$ (2.50) and $D$ (4.61)

**Layer 1** is a sparser subset — here, $\{C, D, F\}$ — each connected to
its nearest neighbor *within that subset*: $C$'s closest layer-1 point is
$D$ (5.66), and $D$'s closest layer-1 point is $F$ (4.61). This forms a
short chain $C - D - F$.

**Layer 2**, the top layer, contains a single entry point: $C$. Notice $C$
is the *farthest* point from $Q$ in this example (tied with $A$ at 7.649) —
deliberately, so the search has real distance to cover, just like a highway
entry ramp might be far from your final destination as the crow flies.

In [4]:
# LAYERS[layer][node] = set of neighbors of `node` within that layer.
# Layer 0: every node connected to its 2 nearest neighbors (from the table above).
# Layer 1: sparse subset {C, D, F}, chained by nearest-within-subset.
# Layer 2: single entry point C.
LAYERS = {
    0: {
        "A": {"B", "F"},
        "B": {"D", "A"},
        "C": {"D", "E"},
        "D": {"B", "E"},
        "E": {"F", "D"},
        "F": {"E", "D"},
    },
    1: {"C": {"D"}, "D": {"C", "F"}, "F": {"D"}},
    2: {"C": set()},
}


def greedy_search_layer(layer, entry):
    """Greedy walk within one HNSW layer.

    Repeatedly move to whichever neighbor of the current node is closest to
    the query, until no neighbor improves on the current node. Returns the
    final node and its distance to the query.
    """
    current = entry
    current_dist = dist_to_query(current)
    print(f"    start at {current} (dist={current_dist:.3f})")
    while True:
        candidates = [(n, dist_to_query(n)) for n in LAYERS[layer][current]]
        for n, d in candidates:
            print(f"    check neighbor {n} (dist={d:.3f})")
        if not candidates:
            print("    no neighbors at this layer -> stop")
            return current, current_dist
        best_node, best_dist = min(candidates, key=lambda c: c[1])
        if best_dist < current_dist:
            print(f"    -> move to {best_node} (dist={best_dist:.3f} < {current_dist:.3f})")
            current, current_dist = best_node, best_dist
        else:
            print(f"    -> no neighbor closer than {current} ({current_dist:.3f}); stop")
            return current, current_dist


print("=== HNSW greedy search for Q = (7.5, 1.5) ===")
print("\nLayer 2 (entry point):")
current, current_dist = "C", dist_to_query("C")
print(f"  entry = {current} (dist={current_dist:.3f}), no neighbors at layer 2 -> drop down")

print("\nLayer 1:")
current, current_dist = greedy_search_layer(1, current)
print(f"  -> layer 1 result: {current} (dist={current_dist:.3f})")

print("\nLayer 0:")
current, current_dist = greedy_search_layer(0, current)
print(f"  -> layer 0 result: {current} (dist={current_dist:.3f})")

print(f"\nHNSW final answer: {current} (dist={current_dist:.3f})")
true_nn = min(POINTS, key=dist_to_query)
print(f"Brute-force true nearest neighbor: {true_nn} (dist={dist_to_query(true_nn):.3f})")
print(f"Match: {current == true_nn}")

=== HNSW greedy search for Q = (7.5, 1.5) ===

Layer 2 (entry point):
  entry = C (dist=7.649), no neighbors at layer 2 -> drop down

Layer 1:
    start at C (dist=7.649)
    check neighbor D (dist=4.301)
    -> move to D (dist=4.301 < 7.649)
    check neighbor C (dist=7.649)
    check neighbor F (dist=1.803)
    -> move to F (dist=1.803 < 4.301)
    check neighbor D (dist=4.301)
    -> no neighbor closer than F (1.803); stop
  -> layer 1 result: F (dist=1.803)

Layer 0:
    start at F (dist=1.803)
    check neighbor D (dist=4.301)
    check neighbor E (dist=0.707)
    -> move to E (dist=0.707 < 1.803)
    check neighbor D (dist=4.301)
    check neighbor F (dist=1.803)
    -> no neighbor closer than E (0.707); stop
  -> layer 0 result: E (dist=0.707)

HNSW final answer: E (dist=0.707)
Brute-force true nearest neighbor: E (dist=0.707)
Match: True


In this example, the greedy walk visited only 4 of the 6 points
($C \to D \to F \to E$, never touching $A$ or $B$) and still found the true
nearest neighbor $E$. That's the appeal of HNSW: a small fraction of the
work, often the right answer.

But notice *why* it worked here: every step happened to have a neighbor that
was closer to $Q$ than the current node, all the way down to $E$. This isn't
guaranteed. If $E$'s only layer-0 connections were to points *farther* from
$Q$ than $E$ itself (a "local minimum" in the graph), the greedy walk would
stop early and report a wrong answer — even though $E$ is sitting right
there, just not connected to anything the walk visited. This is exactly why
FAISS's `IndexHNSWFlat` in Section 3 doesn't always achieve 100% recall, and
why increasing `efSearch` (which lets the walk keep a *list* of candidates
instead of committing to a single best neighbor) improves recall by reducing
the chance of getting stuck in a local minimum.

## 3. FAISS Hands-On

[FAISS](https://github.com/facebookresearch/faiss) (Facebook AI Similarity
Search) is the library most production vector databases build their
indexing on top of (Chroma's default index is HNSW, for example). Two index
types matter here:

`IndexFlatL2` stores every vector verbatim and answers each query with a
brute-force scan — exactly the numpy loop from Section 1, just implemented
in optimized C++ with SIMD. It's the **ground truth** against which we'll
measure ANN recall.

`IndexHNSWFlat` builds the layered graph from Section 2 automatically. Its
key parameters are `M` (how many neighbors each node connects to per layer —
bigger $M$ means a denser, more accurate but larger graph), `efConstruction`
(how thorough the search is while *building* each node's connections), and
`efSearch` (how many candidates the greedy walk keeps track of at *query*
time — bigger $ef$ means slower but more accurate search, directly
controlling the recall/speed tradeoff from Section 1).

In [5]:
# --- Exact index: ground truth ---
index_flat = faiss.IndexFlatL2(DIM)
index_flat.add(vectors)

start = time.perf_counter()
flat_dists, flat_ids = index_flat.search(queries, K)
flat_elapsed = time.perf_counter() - start
print("--- IndexFlatL2 (exact) ---")
print(f"Search time for {N_QUERIES} queries: {flat_elapsed*1000:.3f} ms "
      f"({flat_elapsed/N_QUERIES*1000:.4f} ms/query)")
print(f"Top-{K} neighbors for query 0: {flat_ids[0]}")

# --- Approximate index: HNSW with M=32 neighbors per layer ---
index_hnsw = faiss.IndexHNSWFlat(DIM, 32)
index_hnsw.hnsw.efConstruction = 40
index_hnsw.add(vectors)
index_hnsw.hnsw.efSearch = 16

start = time.perf_counter()
hnsw_dists, hnsw_ids = index_hnsw.search(queries, K)
hnsw_elapsed = time.perf_counter() - start
print("\n--- IndexHNSWFlat (approximate, M=32, efSearch=16) ---")
print(f"Search time for {N_QUERIES} queries: {hnsw_elapsed*1000:.3f} ms "
      f"({hnsw_elapsed/N_QUERIES*1000:.4f} ms/query)")
print(f"Top-{K} neighbors for query 0: {hnsw_ids[0]}")

# --- Recall@K: how many of FlatL2's true top-K does HNSW also return? ---
recalls = [
    len(set(flat_ids[q]) & set(hnsw_ids[q])) / K
    for q in range(N_QUERIES)
]
print(f"\nRecall@{K} (HNSW vs FlatL2 ground truth): "
      f"mean={np.mean(recalls):.3f}, min={np.min(recalls):.3f}, max={np.max(recalls):.3f}")
print(f"Speedup (flat ms/query / hnsw ms/query): {flat_elapsed / hnsw_elapsed:.2f}x")

--- IndexFlatL2 (exact) ---
Search time for 20 queries: 7.519 ms (0.3760 ms/query)
Top-10 neighbors for query 0: [24762  4281 29230 34062 48673 34101 35600  5390 24329 45167]



--- IndexHNSWFlat (approximate, M=32, efSearch=16) ---
Search time for 20 queries: 4.197 ms (0.2099 ms/query)
Top-10 neighbors for query 0: [ 4281 48673 34101 45167 34148 16313 27095 34658 28325 24995]

Recall@10 (HNSW vs FlatL2 ground truth): mean=0.585, min=0.400, max=0.700
Speedup (flat ms/query / hnsw ms/query): 1.79x


HNSW's first run used `efSearch=16` and recovered roughly 60% of the
true top-10 while running over an order of magnitude faster than the exact
scan. `efSearch` controls how large the candidate list is during the greedy
walk — increasing it lets the search consider more alternatives before
committing, raising recall at the cost of latency. The sweep below traces
out this tradeoff curve directly, on the *same* HNSW index (rebuilding the
graph is expensive; changing `efSearch` only affects the query-time
search).

In [6]:
print(f"{'efSearch':>10} | {'recall@'+str(K):>10} | {'ms/query':>10}")
for ef in [4, 8, 16, 32, 64]:
    index_hnsw.hnsw.efSearch = ef
    start = time.perf_counter()
    _, ids = index_hnsw.search(queries, K)
    elapsed = time.perf_counter() - start
    recalls = [len(set(flat_ids[q]) & set(ids[q])) / K for q in range(N_QUERIES)]
    print(f"{ef:>10} | {np.mean(recalls):>10.3f} | {elapsed/N_QUERIES*1000:>10.4f}")

  efSearch |  recall@10 |   ms/query
         4 |      0.240 |     0.0998
         8 |      0.410 |     0.0181
        16 |      0.585 |     0.0222
        32 |      0.725 |     0.0336
        64 |      0.895 |     0.0509


## 4. Embedded vs Server DBs (Exercise)

FAISS is a *library* — it gives you index data structures and search
algorithms, but no persistence, no metadata, and no client/server
separation. **Vector databases** wrap an ANN index (often FAISS or HNSW
itself) with those production features. Two common deployment models:

**Chroma** is an *embedded* database — it runs in the same process as your
application, backed by SQLite plus a local HNSW index, similar to how SQLite
itself is an embedded relational database. Great for prototyping and
small-to-medium local datasets; no separate server to run.

**Qdrant** is a *server-based* database — normally a separate process (or
container) your application talks to over HTTP/gRPC, though its Python
client also offers an in-memory mode (`QdrantClient(":memory:")`) for
testing, which is what we'll use here. Server-based databases add features
like sharding across machines, replication, and built-in payload (metadata)
filtering — at the cost of operational complexity.

The cell below loads a 2,000-vector subset of the Section 1 dataset — each
vector tagged with its `category` — into both databases. Your exercise is to
write the queries against each.

In [7]:
# Chroma needs sqlite3 >= 3.35.0; the system's sqlite3 is older, so swap in
# the pysqlite3-binary package's module before importing chromadb.
__import__("pysqlite3")
import sys
sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
import chromadb

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

N_SUBSET = 2000
subset_vectors = vectors[:N_SUBSET]
subset_categories = [CATEGORIES[c] for c in category_ids[:N_SUBSET]]
subset_ids = [str(i) for i in range(N_SUBSET)]

print(f"Subset: {N_SUBSET} vectors, dim={DIM}")

# --- Chroma: embedded, in-process ---
chroma_client = chromadb.Client()
chroma_collection = chroma_client.create_collection(name="topic6_demo")
chroma_collection.add(
    ids=subset_ids,
    embeddings=subset_vectors.tolist(),
    metadatas=[{"category": c} for c in subset_categories],
)
print(f"Chroma collection size: {chroma_collection.count()}")

# --- Qdrant: in-memory client (same API as a real server) ---
qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name="topic6_demo",
    vectors_config=VectorParams(size=DIM, distance=Distance.EUCLID),
)
qdrant_client.upsert(
    collection_name="topic6_demo",
    points=[
        PointStruct(id=i, vector=subset_vectors[i].tolist(), payload={"category": subset_categories[i]})
        for i in range(N_SUBSET)
    ],
)
print(f"Qdrant collection size: {qdrant_client.count('topic6_demo').count}")

Subset: 2000 vectors, dim=128


Chroma collection size: 2000


Qdrant collection size: 2000


### Exercise — query both databases

Using `query_vec = queries[0].tolist()` (the first query vector from Section
1), run a top-5 nearest-neighbor query against:

1. `chroma_collection`, using `.query(query_embeddings=[query_vec], n_results=5)`
2. `qdrant_client`, using `.query_points(collection_name="topic6_demo", query=query_vec, limit=5)`

For each, print the returned ids and `category` metadata. Then check whether
the two databases agree on the top-5 ids — both are doing exact search on
2,000 vectors here, so they should.

In [8]:
query_vec = queries[0].tolist()

chroma_result = chroma_collection.query(query_embeddings=[query_vec], n_results=5)
print("Chroma top-5:")
print(f"  ids:        {chroma_result['ids'][0]}")
print(f"  categories: {[m['category'] for m in chroma_result['metadatas'][0]]}")

qdrant_result = qdrant_client.query_points(collection_name="topic6_demo", query=query_vec, limit=5).points
print("\nQdrant top-5:")
print(f"  ids:        {[p.id for p in qdrant_result]}")
print(f"  categories: {[p.payload['category'] for p in qdrant_result]}")

chroma_ids = {int(i) for i in chroma_result["ids"][0]}
qdrant_ids = {p.id for p in qdrant_result}
print(f"\nSame top-5 ids: {chroma_ids == qdrant_ids}")

Chroma top-5:
  ids:        ['1629', '1773', '807', '1112', '1431']
  categories: ['cooking', 'cooking', 'cooking', 'cooking', 'cooking']

Qdrant top-5:
  ids:        [1629, 1773, 807, 1112, 1431]
  categories: ['cooking', 'cooking', 'cooking', 'cooking', 'cooking']

Same top-5 ids: True


## 5. Metadata Filtering (Exercise)

Real queries are rarely "find the 5 nearest vectors" in isolation — they're
"find the 5 nearest vectors *where category = 'biology'*" (or `user_id =
42`, or `date > last_week`). There are two naive ways to combine a vector
search with a metadata filter, and one better way:

**Post-filtering** runs the ANN search first (e.g. top-5 by vector
similarity), then discards results that don't match the filter. This is
simple but can return *fewer results than requested* — or zero — if none of
the nearest vectors happen to match the filter, even when plenty of matching
vectors exist elsewhere in the index.

**Pre-filtering** restricts the search to only the vectors matching the
filter *before* the ANN traversal — e.g. by computing the candidate set
first and brute-force searching just that subset. This always returns up to
the requested number of results (if that many matches exist), but can be
slow if the filter is not very selective and the candidate set is large.

**Filterable HNSW** (Qdrant's actual approach) integrates the filter into
the graph traversal itself: the greedy walk only considers neighbors that
pass the filter, skipping non-matching nodes without abandoning the graph
structure. This gets pre-filtering's correctness without pre-filtering's
full-scan cost. Qdrant's `query_filter` argument implements this.

### Exercise — reproduce the post-filtering failure mode

1. Run an *unfiltered* `qdrant_client.query_points(collection_name="topic6_demo", query=query_vec, limit=5)` and print each result's `category`.
2. Pick `TARGET_CATEGORY` to be a category that does **not** appear in that unfiltered top-5.
3. **Post-filter**: from that same unfiltered top-5, keep only results where `category == TARGET_CATEGORY`, and print how many remain.
4. **Pre-filter**: call `qdrant_client.query_points(collection_name="topic6_demo", query=query_vec, query_filter=Filter(must=[FieldCondition(key="category", match=MatchValue(value=TARGET_CATEGORY))]), limit=5)` and print the results.
5. Compare the two result counts.

In [9]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

unfiltered = qdrant_client.query_points(collection_name="topic6_demo", query=query_vec, limit=5).points
print("Unfiltered top-5:")
for p in unfiltered:
    print(f"  id={p.id:5d}  category={p.payload['category']:8s}  score={p.score:.4f}")

unfiltered_categories = {p.payload["category"] for p in unfiltered}
TARGET_CATEGORY = next(c for c in CATEGORIES if c not in unfiltered_categories)
print(f"\nTARGET_CATEGORY = '{TARGET_CATEGORY}' (absent from unfiltered top-5: {unfiltered_categories})")

post_filtered = [p for p in unfiltered if p.payload["category"] == TARGET_CATEGORY]
print(f"\nPost-filter result count: {len(post_filtered)} (requested 5)")

pre_filtered = qdrant_client.query_points(
    collection_name="topic6_demo", query=query_vec,
    query_filter=Filter(must=[FieldCondition(key="category", match=MatchValue(value=TARGET_CATEGORY))]),
    limit=5,
).points
print(f"\nPre-filter result count: {len(pre_filtered)} (requested 5)")
for p in pre_filtered:
    print(f"  id={p.id:5d}  category={p.payload['category']:8s}  score={p.score:.4f}")

Unfiltered top-5:
  id= 1629  category=cooking   score=7.9029
  id= 1773  category=cooking   score=7.9700
  id=  807  category=cooking   score=7.9819
  id= 1112  category=cooking   score=8.0150
  id= 1431  category=cooking   score=8.0423

TARGET_CATEGORY = 'physics' (absent from unfiltered top-5: {'cooking'})

Post-filter result count: 0 (requested 5)

Pre-filter result count: 5 (requested 5)
  id=  296  category=physics   score=18.8173
  id=  727  category=physics   score=18.8318
  id=  244  category=physics   score=18.9898
  id= 1546  category=physics   score=19.0688
  id=  950  category=physics   score=19.1253


## 6. Vector Quantization for Storage

Every index so far has stored each vector as 128 float32 numbers — 512
bytes per vector, or about 25.6 MB for our 50,000-vector dataset. At
billions of vectors, raw float32 storage becomes a major memory cost, often
*larger* than the graph structure itself. **Vector quantization** compresses
each stored vector into a smaller code, trading some accuracy for a much
smaller index.

**Scalar quantization (SQ8)** maps each of the 128 float32 dimensions
independently to a single byte (256 levels), based on the observed value
range of that dimension across the dataset — a straightforward $4\times$
compression.

**Product quantization (PQ)** splits each vector into $M$ sub-vectors (e.g.
128 dimensions into 16 chunks of 8 dimensions each), and separately runs
k-means on each sub-vector's values across the whole dataset to learn 256
"centroid" codes per chunk. Each stored vector then becomes just $M$
single-byte centroid indices — for $M=16$, that's 16 bytes per vector, a
$32\times$ compression. Smaller $M$ means more compression but coarser
sub-vectors (fewer dimensions' worth of information per code), which
typically hurts recall more.

The Topic 9 connection: this is the *exact same idea* as quantizing a neural
network's weights from float32 to int8 — replace high-precision numbers with
a small lookup index into a learned codebook, accepting a controlled
accuracy loss for a large memory win.

In [10]:
print(f"Dataset: {N_VECTORS} vectors, dim={DIM}")
flat_bytes_per_vector = DIM * 4  # float32
print(f"IndexFlatL2 baseline:   {flat_bytes_per_vector:4d} bytes/vector  (recall@{K} = 1.000)")

# --- Scalar quantization: each float32 dimension -> 1 byte ---
index_sq = faiss.IndexScalarQuantizer(DIM, faiss.ScalarQuantizer.QT_8bit)
index_sq.train(vectors)
index_sq.add(vectors)
_, sq_ids = index_sq.search(queries, K)
sq_recalls = [len(set(flat_ids[q]) & set(sq_ids[q])) / K for q in range(N_QUERIES)]
sq_bytes_per_vector = DIM * 1
print(f"IndexScalarQuantizer:   {sq_bytes_per_vector:4d} bytes/vector  "
      f"({flat_bytes_per_vector / sq_bytes_per_vector:.1f}x smaller)  "
      f"recall@{K} = {np.mean(sq_recalls):.3f}")

# --- Product quantization: split DIM into M sub-vectors, 8-bit code each ---
NBITS = 8
print("\nProduct quantization (PQ): more sub-vectors M = finer codes = better recall, less compression")
for M in [8, 16, 32, 64]:
    index_pq = faiss.IndexPQ(DIM, M, NBITS)
    index_pq.train(vectors)
    index_pq.add(vectors)
    _, pq_ids = index_pq.search(queries, K)
    pq_recalls = [len(set(flat_ids[q]) & set(pq_ids[q])) / K for q in range(N_QUERIES)]
    pq_bytes_per_vector = M * NBITS // 8
    print(f"  IndexPQ (M={M:2d}): {pq_bytes_per_vector:4d} bytes/vector  "
          f"({flat_bytes_per_vector / pq_bytes_per_vector:5.1f}x smaller)  "
          f"recall@{K} = {np.mean(pq_recalls):.3f}")

print(f"\nTotal index size for all {N_VECTORS} vectors:")
print(f"  IndexFlatL2:             {N_VECTORS * flat_bytes_per_vector / 1024:8.1f} KB")
print(f"  IndexScalarQuantizer:    {N_VECTORS * sq_bytes_per_vector / 1024:8.1f} KB")
print(f"  IndexPQ (M=64):          {N_VECTORS * pq_bytes_per_vector / 1024:8.1f} KB")

Dataset: 50000 vectors, dim=128
IndexFlatL2 baseline:    512 bytes/vector  (recall@10 = 1.000)
IndexScalarQuantizer:    128 bytes/vector  (4.0x smaller)  recall@10 = 0.980

Product quantization (PQ): more sub-vectors M = finer codes = better recall, less compression


  IndexPQ (M= 8):    8 bytes/vector  ( 64.0x smaller)  recall@10 = 0.055


  IndexPQ (M=16):   16 bytes/vector  ( 32.0x smaller)  recall@10 = 0.145


  IndexPQ (M=32):   32 bytes/vector  ( 16.0x smaller)  recall@10 = 0.350


  IndexPQ (M=64):   64 bytes/vector  (  8.0x smaller)  recall@10 = 0.760

Total index size for all 50000 vectors:
  IndexFlatL2:              25000.0 KB
  IndexScalarQuantizer:      6250.0 KB
  IndexPQ (M=64):            3125.0 KB


## Putting It All Together

| Tool | Persistence | Metadata filtering | Scaling | Ease of setup |
|---|---|---|---|---|
| FAISS | None built-in (manual save/load of index files) | None — pure vector index | Single machine; very large indices via PQ/IVF | `pip install faiss-cpu`, pure library |
| Chroma | Local on-disk (SQLite + index files), embedded | Yes — `where` filters on metadata | Single machine | `pip install chromadb`, runs in-process |
| Qdrant | Server-managed (disk-backed collections), or in-memory for testing | Yes — native filterable HNSW (`query_filter`) | Horizontal — sharding & replication across nodes | Run as a server (Docker), or `:memory:` client for testing |

## Where to Go Next

Topic 7 (Structured Outputs & Tool Calling) moves up a layer: now that
retrieval can return the right chunks efficiently and with the right
filters, how do you get an LLM to reliably consume them and produce
machine-parseable output — the foundation for the agentic tool-calling loops
from Topic 4.